# Demo — Adapting TissueTypist to lung tissue (LOSO accuracy)

This notebook adapts TissueTypist to a **non-cardiac, non-imaging**
dataset: human lung Visium SD 3′ from
[Madissoon et al., 2023](https://doi.org/10.1038/s41588-022-01243-4).
The point is to show two things at once:

1. **TissueTypist generalises.** The hierarchy framework, training
   pipeline, and prediction code are tissue-agnostic — none of the
   cardiac vocabulary leaks through. A single CLI flag (`--flat`)
   trains a classifier on whatever label column you have.
2. **Cross-section accuracy is honest.** With 11 lung sections we run
   a full leave-one-section-out (LOSO) loop and report per-fold +
   pooled F1.

**Setup:**

- Single modality (Visium SD 3′) → only `--reference` is passed.
- Flat label column (`tissue_label`) → `--flat --coarse_col tissue_label`,
  no sub-models, no YAML.
- Mac-friendly compute → `--neighbour_weight 0 --edge_weight 0`
  (`own_only` weights). Skips KNN + edge-distance, so each fold trains
  on gene expression alone. Drop the flags for the cardiac default
  (0.3 / 5.0) if you have a workstation.

**Data quirks of this dataset (handled inline):**

- `obs["sample"]` is the section identifier — TissueTypist hardcodes
  `section_ID` for training, so we copy the column once at setup.
- `adata.X` is already log-normalised (the `_preprocessed` filename
  hints at this). `normalise_if_needed` detects and skips re-normalising.


## 1. Setup


In [ ]:
# Edit these paths if your local layout differs.
from pathlib import Path

REPO       = Path.home() / "GitHub" / "TissueTypist"
LUNG_QUERY = Path.home() / "GitHub" / "anndata" / "Madissoon2023_lung_visium_preprocessed.h5ad"

LOSO_OUTDIR = REPO / "results" / "lung_loso_demo"      # per-fold cache + pooled outputs
LOSO_OUTDIR.mkdir(parents=True, exist_ok=True)

assert LUNG_QUERY.exists(), f"Missing input: {LUNG_QUERY}"


In [ ]:
import json, subprocess, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

import tissuetypist
from tissuetypist import predict_adata
print("TissueTypist:", getattr(tissuetypist, "__version__", "(dev install)"))


## 2. Inspect the lung dataset

Loading once. The remaining cells reuse this `adata` — no re-reads.


In [ ]:
adata = sc.read_h5ad(LUNG_QUERY)

# TissueTypist's training code reads `obs["section_ID"]` directly. Copy
# `sample` over so we don't have to fork the upstream code path.
adata.obs["section_ID"] = adata.obs["sample"].astype(str)

print(adata)
print()
print("Tissue labels:", adata.obs["tissue_label"].nunique())
print("Sections     :", adata.obs["section_ID"].nunique())


In [ ]:
# How many spots per tissue label?
adata.obs["tissue_label"].value_counts()


In [ ]:
# How are labels distributed across sections? Useful for spotting
# rare-class folds before kicking off the loop.
xtab = pd.crosstab(adata.obs["tissue_label"], adata.obs["section_ID"])
xtab


In [ ]:
# Quick spatial preview of a single section.
sec = adata.obs["section_ID"].iloc[0]
sub = adata[adata.obs["section_ID"] == sec]

palette = sns.color_palette("tab20", n_colors=adata.obs["tissue_label"].nunique())
label_to_colour = dict(zip(sorted(adata.obs["tissue_label"].unique()), palette))

fig, ax = plt.subplots(figsize=(6, 6))
xy = sub.obsm["spatial"]
colours = [label_to_colour[l] for l in sub.obs["tissue_label"]]
ax.scatter(xy[:, 0], xy[:, 1], c=colours, s=6, linewidths=0)
ax.set_title(f"Section {sec} — ground-truth tissue_label")
ax.set_aspect("equal"); ax.axis("off")
ax.invert_yaxis()  # match Visium image orientation
plt.show()


## 3. Leave-one-section-out validation

TissueTypist's accuracy in the manuscript is reported as LOSO across
14 cardiac sections. The same protocol works here: 11 lung sections,
each held out in turn, so the model is always asked to generalise to
*sample geometry it has never seen*.

For every fold we:

1. Subset `adata` to the 10 training sections, write a temp h5ad.
2. Run `tissuetypist train --flat` on it (own_only weights — fast).
3. Run `predict_adata` on the held-out section.
4. Score predictions against `tissue_label` (weighted + macro F1).
5. Cache predictions and metrics under `LOSO_OUTDIR / per_fold / <section>/`.

The cache makes the loop **resumable** — if the kernel crashes mid-loop,
re-running picks up where it left off.


## 4. One fold, step by step

Walk through a single fold first so you can see what each step
produces. The full loop in §5 is the same code wrapped in a `for`.


In [ ]:
def train_one_fold(adata, held_out_section, out_root):
    # Train on all sections != held_out_section. Returns model_dir path.
    fold_dir = Path(out_root) / "per_fold" / f"{held_out_section}"
    model_dir = fold_dir / "model"
    if (model_dir / "hierarchy_config.json").exists():
        return model_dir   # already trained

    fold_dir.mkdir(parents=True, exist_ok=True)
    train_mask = adata.obs["section_ID"] != held_out_section
    train_adata = adata[train_mask].copy()

    train_path = fold_dir / "train_subset.h5ad"
    train_adata.write_h5ad(train_path)

    cmd = [
        "tissuetypist", "train",
        "--reference",       str(train_path),
        "--outdir",          str(model_dir),
        "--flat",            "--coarse_col", "tissue_label",
        "--neighbour_weight", "0",
        "--edge_weight",      "0",
    ]
    print("$ " + " ".join(cmd))
    subprocess.run(cmd, check=True)

    train_path.unlink()    # don't leave 200+ MB temp files lying around
    return model_dir


In [ ]:
# Pick the first section as the demo fold.
demo_section = sorted(adata.obs["section_ID"].unique())[0]
print(f"Held out: {demo_section}")

demo_model_dir = train_one_fold(adata, demo_section, LOSO_OUTDIR)
print("Model dir:", demo_model_dir)


In [ ]:
# Predict on the held-out section.
test_adata = adata[adata.obs["section_ID"] == demo_section].copy()
test_adata = predict_adata(
    test_adata,
    model_dir   = str(demo_model_dir),
    modality    = "sd",
    section_col = "section_ID",
)

test_adata.obs[["tissue_label", "tt_final_label", "tt_coarse_score"]].head()


In [ ]:
from sklearn.metrics import f1_score, classification_report

y_true = test_adata.obs["tissue_label"].astype(str)
y_pred = test_adata.obs["tt_final_label"].astype(str)

print(f"Held-out section: {demo_section}  (n_spots = {len(y_true):,})")
print(f"  F1 weighted: {f1_score(y_true, y_pred, average='weighted'):.3f}")
print(f"  F1 macro   : {f1_score(y_true, y_pred, average='macro'):.3f}")
print()
print(classification_report(y_true, y_pred, zero_division=0))


## 5. Full 11-fold loop

Per-fold caching: each fold writes `predicted.h5ad` + `metrics.json`
under `per_fold/<section>/`, and on re-run we skip any fold that already
has both files. So if you Ctrl+C halfway through, the next run finishes
the rest.

> **Heads up.** With `own_only` weights, expect ~3–10 min per fold on a
> Mac depending on spot count. Total ~30 min – 2 h. Open a terminal and
> `caffeinate` if you want to leave it running.


In [ ]:
def _sanitise_obs_for_h5ad(adata):
    # Flat hierarchies leave `tt_fine_label` / `tt_fine_score` / `tt_joint_score`
    # all-None — h5ad's vlen-string writer can't serialise that. Drop fully-null
    # tt_* columns and cast partially-filled object columns to str. This is
    # purely a write-time fix; the predictions themselves are fine in memory.
    for col in list(adata.obs.columns):
        s = adata.obs[col]
        if s.dtype != object:
            continue
        if s.isna().all():
            del adata.obs[col]
        else:
            adata.obs[col] = s.fillna("").astype(str)
    return adata


def run_one_fold(adata, held_out_section, out_root):
    # Train + predict + score one fold. Idempotent (uses cache).
    fold_dir = Path(out_root) / "per_fold" / f"{held_out_section}"
    pred_path = fold_dir / "predicted.h5ad"
    metrics_path = fold_dir / "metrics.json"

    if pred_path.exists() and metrics_path.exists():
        return json.loads(metrics_path.read_text())

    model_dir = train_one_fold(adata, held_out_section, out_root)

    test_adata = adata[adata.obs["section_ID"] == held_out_section].copy()
    test_adata = predict_adata(
        test_adata, model_dir=str(model_dir),
        modality="sd", section_col="section_ID",
    )
    _sanitise_obs_for_h5ad(test_adata)
    test_adata.write_h5ad(pred_path)

    y_true = test_adata.obs["tissue_label"].astype(str)
    y_pred = test_adata.obs["tt_final_label"].astype(str)
    metrics = {
        "section":     str(held_out_section),
        "n_spots":     int(len(y_true)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted",
                                      zero_division=0)),
        "f1_macro":    float(f1_score(y_true, y_pred, average="macro",
                                      zero_division=0)),
    }
    metrics_path.write_text(json.dumps(metrics, indent=2))
    return metrics


In [ ]:
sections = sorted(adata.obs["section_ID"].unique())
print(f"LOSO over {len(sections)} sections.")

all_metrics = []
for i, sec in enumerate(sections, 1):
    print(f"[{i:2d}/{len(sections)}] Held-out: {sec}")
    m = run_one_fold(adata, sec, LOSO_OUTDIR)
    print(f"           n={m['n_spots']:>5,}  "
          f"F1_w={m['f1_weighted']:.3f}  F1_m={m['f1_macro']:.3f}")
    all_metrics.append(m)


## 6. Aggregate per-fold + pooled metrics

Two ways to summarise LOSO accuracy:

- **Per-fold + mean** — each section is a single-experiment estimate;
  the mean across folds is the cross-section generalisation.
- **Pooled** — concatenate all held-out predictions, score once.
  Equivalent to a weighted mean (weighted by spot count).

Both are reported below.


In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(LOSO_OUTDIR / "per_fold_metrics.csv", index=False)
metrics_df


In [ ]:
print("Per-fold mean ± std (unweighted across folds)")
print(f"  F1 weighted: {metrics_df['f1_weighted'].mean():.3f} ± {metrics_df['f1_weighted'].std():.3f}")
print(f"  F1 macro   : {metrics_df['f1_macro'].mean():.3f} ± {metrics_df['f1_macro'].std():.3f}")


In [ ]:
# Pooled across folds — concatenate every held-out section's predictions.
pooled_pieces = []
for sec in sections:
    p = LOSO_OUTDIR / "per_fold" / sec / "predicted.h5ad"
    a = sc.read_h5ad(p)
    pooled_pieces.append(pd.DataFrame({
        "section":  sec,
        "y_true":   a.obs["tissue_label"].astype(str).values,
        "y_pred":   a.obs["tt_final_label"].astype(str).values,
        "tt_coarse_score": a.obs["tt_coarse_score"].astype(float).values,
    }))
pooled = pd.concat(pooled_pieces, ignore_index=True)
pooled.to_parquet(LOSO_OUTDIR / "pooled_predictions.parquet")

print(f"Pooled predictions: {len(pooled):,} spots across {pooled['section'].nunique()} sections")
print(f"  F1 weighted: {f1_score(pooled['y_true'], pooled['y_pred'], average='weighted', zero_division=0):.3f}")
print(f"  F1 macro   : {f1_score(pooled['y_true'], pooled['y_pred'], average='macro',    zero_division=0):.3f}")


## 7. Per-tissue-label F1

Coarse weighted F1 doesn't tell you which classes are hard. Here's a
per-class F1, computed on the pooled predictions, plus a barplot
ranked by score.


In [ ]:
from sklearn.metrics import f1_score
labels = sorted(set(pooled["y_true"]) | set(pooled["y_pred"]))
per_label = pd.DataFrame({
    "tissue_label": labels,
    "f1":           [f1_score(pooled["y_true"], pooled["y_pred"],
                              labels=[l], average="macro", zero_division=0)
                     for l in labels],
    "n_spots_gt":   [int((pooled["y_true"] == l).sum()) for l in labels],
}).sort_values("f1", ascending=False)
per_label.to_csv(LOSO_OUTDIR / "per_label_f1.csv", index=False)
per_label


In [ ]:
fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(per_label))))
ax.barh(per_label["tissue_label"], per_label["f1"],
        color=[label_to_colour[l] for l in per_label["tissue_label"]])
ax.set_xlabel("F1 (pooled across LOSO folds)")
ax.set_xlim(0, 1)
ax.invert_yaxis()
for i, (_, row) in enumerate(per_label.iterrows()):
    ax.text(row["f1"] + 0.01, i, f"n={row['n_spots_gt']:,}",
            va="center", fontsize=8)
ax.set_title("Per-tissue-label F1 — lung LOSO (own_only weights)")
plt.tight_layout()
plt.show()


## 8. Where are the errors?

A pooled confusion matrix, plus a side-by-side ground-truth vs
prediction map for one section. Helps spot whether errors are mostly
*confusions between adjacent niches* (often acceptable) or
*structurally wrong* (a real failure).


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(pooled["y_true"], pooled["y_pred"], labels=labels,
                      normalize="true")
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_df, cmap="Blues", vmin=0, vmax=1, square=True,
            cbar_kws={"label": "Recall"}, ax=ax)
ax.set_xlabel("Predicted (tt_final_label)")
ax.set_ylabel("Ground truth (tissue_label)")
ax.set_title("Pooled confusion matrix — lung LOSO")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side: GT vs prediction for one held-out section.
sec = demo_section
a = sc.read_h5ad(LOSO_OUTDIR / "per_fold" / sec / "predicted.h5ad")
xy = a.obsm["spatial"]

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, col, title in [
    (axes[0], "tissue_label",   f"Ground truth — {sec}"),
    (axes[1], "tt_final_label", f"TissueTypist (LOSO) — {sec}"),
]:
    colours = [label_to_colour.get(l, "#cccccc") for l in a.obs[col].astype(str)]
    ax.scatter(xy[:, 0], xy[:, 1], c=colours, s=6, linewidths=0)
    ax.set_title(title)
    ax.set_aspect("equal"); ax.axis("off")
    ax.invert_yaxis()

# Single legend across both panels.
import matplotlib.patches as mpatches
present = sorted(set(a.obs["tissue_label"].astype(str))
                 | set(a.obs["tt_final_label"].astype(str)))
handles = [mpatches.Patch(color=label_to_colour.get(l, "#cccccc"), label=l)
           for l in present]
fig.legend(handles=handles, loc="lower center", ncol=min(5, len(present)),
           fontsize=8, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()


## 9. Takeaways + next steps

**What this demo shows.** TissueTypist's training and prediction code
are tissue-agnostic — `--flat --coarse_col tissue_label` is enough to
adapt to a new dataset. The 11-fold LOSO gives an honest estimate of
cross-section accuracy, which is the question that matters when you
plan to apply the model to new patients / sections.

**When to graduate from `--flat`.**

- *You have a coarse + fine label hierarchy*. Switch to
  `tissuetypist train --auto_infer --coarse_col COARSE --fine_col FINE`.
  This builds one flat sub-model per coarse niche (so e.g. "Airway"
  and "Vasculature" each get their own fine-grained classifier).
- *Your hierarchy has 3+ levels, modality-specific stages, pooled
  intermediate labels, or a custom palette*. Write a YAML — see
  [docs/hierarchy.md](../hierarchy.md) for the schema and copy
  `tissuetypist/config/hierarchies/cardiac.yaml` as a starting
  template. Validate before a long run with
  `tissuetypist validate-hierarchy my_lung.yaml --adata my_data.h5ad`.

**When to graduate from `own_only` weights.**

- *You expect tissue architecture to be locally organised* (e.g.
  airway-epithelium spots are surrounded by other epithelium spots).
  Drop the `--neighbour_weight 0 --edge_weight 0` flags to get the
  cardiac default (0.3 / 5.0). Each fold gets slower because of the
  KNN + edge-distance computation, but pooled F1 typically goes up by
  1–5 points.
- *You want to disentangle "spatial features helping" from "section
  coverage helping"*. Run both `default` and `own_only` LOSO and
  compare per-class. The cardiac manuscript does this in
  `notebooks/three_way_comparison.ipynb`.

**What to read next.**

- [docs/user-guide.md](../user-guide.md) — every workflow including
  panel-specific retraining (Xenium / MERFISH / CosMx), evaluation
  plots, and shipped weight presets.
- [docs/hierarchy.md](../hierarchy.md) — the YAML hierarchy schema.
- [docs/output-columns.md](../output-columns.md) — every `tt_*` column.
- [docs/examples/demo_merfish.ipynb](demo_merfish.ipynb) — the
  imaging-based ST counterpart of this demo.
